In [ ]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
run_on_colab = True
try:
    from google.colab import files, drive
except:
    run_on_colab = False
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from keras.metrics import AUC
from tcn import TCN

from sklearn.preprocessing import LabelEncoder

import keras



In [10]:
# check gpu availability

import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [11]:
if run_on_colab:
    uploaded = files.upload()
    df = pd.read_excel(next(iter(uploaded)))
else:
    df = pd.read_excel('../../data/DataBaseTCN.xlsx')

In [12]:
def plot_save(history, directory_name, save_eps = False) -> None:
    """
    Plot the training and validation metrics.
    Args:
        history (History): Keras History object containing training history.
        metric (str): Metric to plot (e.g., 'accuracy', 'loss').
        title (str): Title of the plot.
    """
    os.makedirs(directory_name, exist_ok=True)
    print("Plotting training history")
    score_names = list(history.history.keys())
    count_perfomance_scores = len(score_names) // 2
    epochs = range(1, dict_hyperparams["epochs"] + 1)
    print(f"epochs: {epochs}")
    for i in range(count_perfomance_scores):
        plt.figure(figsize=(12, 6))
        score_name = score_names[i]
        if score_name == "f1_score":  # f1 score is reported for each class, the plot is hairy
            continue
        plt.xticks(epochs)  # only show integer ticks
        plt.plot(epochs, history.history[score_names[i]], label=score_names[i])
        plt.plot(epochs, history.history[score_names[i + count_perfomance_scores]], 
                label=score_names[i + count_perfomance_scores])
        plt.title(f"Training vs validation {score_names[i]}")
        plt.ylabel(score_names[i])
        plt.xlabel("Epoch")
        plt.legend()
        plt.savefig(f'{os.path.join(directory_name, score_name)}.png', bbox_inches='tight')
        if save_eps:
            plt.savefig(f'{os.path.join(directory_name, score_name)}.eps', bbox_inches='tight')
        plt.show()

In [13]:
dict_hyperparams = {
    "lookback_window":25,  # Time steps used to look back
    "batch_size":32, # 32, 64, 100, ...
    "epochs": 1, # 10, 20, 50, ...
    # "learning_rate": 0.01, # 0.001, 0.01, 0.1, ...
    "nb_filters": 64,
    "nb_stacks": 10, # 1, 2, 3, 
    "kernel_size": 3,
    "use_batch_norm": True, # True, False
    "activation": "relu", # 'relu' 'tanh' 'sigmoid' 'softmax' 'elu' 'selu' 'swish' 'mish'
    "optimizer": "adamw",  # 'sgd' 'rmsprop' 'adam' 'adamw' 'adadelta' 'adagrad' 'adamax' 'adafactor' 'nadam' 'ftrl' 'lion' 'lamb' 'lossscaleoptimizer'
    }

In [14]:
    
def write_predictions(model, x, y_groudtruth, original_labels, dir_name, subset_type):
    full_name = os.path.join(dir_name, f"{subset_type}_predictions.xlsx")
    predicted = model.predict(x)
    df_test = pd.DataFrame(x.reshape(x.shape[0], -1))
    df_test["predicted"] = np.argmax(predicted, axis=1)
    df_test["y_groundtruth"] = np.argmax(y_groudtruth, axis=1)
    df_test.to_excel(full_name, index=False)

In [15]:
def play(df: pd.DataFrame, 
         verbose: bool = False,
         save_eps: bool = False,
         **hyperparameters: str|int|float|bool) -> None: # TODO add proper type hints
    """
    Play with the data and train a TCN model.
    Args:
        df (pd.DataFrame): DataFrame containing the data.
        verbose (bool): If True, print additional information on data
        save_eps (bool): If True, save plots in EPS format as well. The plots are saved as png anyway.
        **kwargs: Additional arguments for the TCN model and data loopback.
    """
    print("Play function called with hyperparameters:", hyperparameters)

    dir_root_results = "results"
    
    timestamp = str(pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M-%S"))
    dir_name = os.path.join(dir_root_results, f"results_{timestamp}")
    os.makedirs(dir_name, exist_ok=True)
    
    # writes hyperparameters
    with open(os.path.join(dir_name, "hyperparameters.txt"), "wt") as f:
        f.write("Hyperparameters:\n")
        for key, value in dict_hyperparams.items():
            f.write(f"\t{key}: {value}\n")
    
    # data preprocessing
    output_column = "Task"  # Column to be used for classification
    df[output_column] = df[output_column].fillna("OPR")  # Fill NaN values with "opr"

    # Define features (excluding label column)
    features = df.drop(columns=["Task"]).values

    if verbose:
        # Print unique text strings
        # Get unique values
        unique_text = df[output_column].dropna().unique()
        print(f"Unique text strings in {output_column}:", unique_text)

        print(df.head())  # Shows first few rows, including header row
        print("Column names:", df.columns)  # Displays detected headers
        
        # find rows with NaN values in column Task
        nan_rows = df[df[output_column].isna()]
        print(f"Rows with NaN values in column {output_column}:")
        print('rows with nans:\n', nan_rows)
        
    # Encode labels if they are categorical
    label_encoder = LabelEncoder()
    # Fit the label encoder to the 'Task' column and transform it to numerical labels
    encoded_labels = label_encoder.fit_transform(df["Task"])
    # Now use the encoded labels for one-hot encoding
    labels = to_categorical(encoded_labels)

    if verbose:
        print("Feature shape:", features.shape)
        print("Label shape:", labels.shape)
        print("Labels:", encoded_labels)
        
    # Create sequences
    x, y = [], []
    lookback_window = hyperparameters["lookback_window"]
    for i in range(lookback_window, len(labels)):
        x.append(features[i - lookback_window:i])
        y.append(labels[i])
        
    x, y = np.array(x), np.array(y)

    # Split into train, validation, and test sets
    x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.3, random_state=42)
    x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

    if verbose:
        print(f'{x_train.shape=}, {y_train.shape=}')
        print(f'{x_val.shape=}, {y_val.shape=}')
        print(f'{x_test.shape=}, {y_test.shape=}')

    # Define TCN model    
    tensorboard_callback = keras.callbacks.TensorBoard(log_dir="./runs")
    i = Input(shape=(lookback_window, features.shape[1]))
    m = TCN(
        use_batch_norm=hyperparameters["use_batch_norm"],
        nb_stacks=hyperparameters["nb_stacks"],
        nb_filters=hyperparameters["nb_filters"],
        kernel_size=hyperparameters["kernel_size"],
        activation = hyperparameters["activation"],
        )(i)
    m = Dense(labels.shape[1], activation="softmax")(m)
    model = Model(inputs=i, outputs=m)

    # run the model
    metrics = [
            "accuracy", 
            "precision", 
            "recall", 
            "f1_score", 
            AUC(multi_label=True, num_labels=labels.shape[1])
            ]
    model.compile(optimizer=hyperparameters["optimizer"],
                loss="categorical_crossentropy", 
                metrics= metrics
                ,
                jit_compile=False)

    # to make callbacks work, use the fix from https://stackoverflow.com/questions/71935007/valueerror-expected-scalar-shape-saw-shape-1
    # example of path: ~/anaconda3/envs/keras/lib/python3.12/site-packages/tensorboard/plugins/scalar/summary_v2.py
    history = model.fit(x_train, y_train, 
                        validation_data=(x_val, y_val), 
                        epochs=hyperparameters["epochs"],
                        batch_size=hyperparameters["batch_size"],
                        callbacks=[tensorboard_callback])

    plot_save(history, dir_name, save_eps=save_eps)
    
    # Predictions
    y_pred_test = model.predict(x_test)
    y_pred_classes_test = np.argmax(y_pred_test, axis=1)
    y_true_classes_test = np.argmax(y_test, axis=1)

    # Reverse encoding to get original names
    original_labels = label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))
    print("Original Labels:", original_labels)

    # Classification Report
    print("Classification Report:\n", classification_report(y_true_classes_test, y_pred_classes_test))
    with open(os.path.join(dir_name, "classification_report.txt"), "wt") as f:
        f.write("Classification Report:")
        f.write(classification_report(y_true_classes_test, y_pred_classes_test))

    # Confusion Matrix
    cm = confusion_matrix(y_true_classes_test, y_pred_classes_test)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, cmap="Reds", fmt="d", xticklabels=original_labels, yticklabels=original_labels)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix on Test Data")
    _confusion_matrix_filename = "confusion_matrix"
    plt.savefig(f'{os.path.join(dir_name, _confusion_matrix_filename)}.png', bbox_inches='tight')
    if save_eps:
        plt.savefig(f'{os.path.join(dir_name, _confusion_matrix_filename)}.eps', bbox_inches='tight')
    plt.show()
    
    # append to previous results
    all_results_path = os.path.join(dir_root_results, "all_results.xlsx")
    # if non existent, create the file and write the header
    if os.path.exists(all_results_path):
        df = pd.read_excel(all_results_path)
    else:
        # add header to the file
        lst_header = ["timestamp"]
        for key in dict_hyperparams.keys():
            lst_header.append(key)
        lst_header += list(history.history.keys())
        df = pd.DataFrame(columns=lst_header)
    
    lst_row = [timestamp]
    for key in dict_hyperparams.keys():
        lst_row.append(str(dict_hyperparams[key]))
    for metric in history.history.keys():
        lst_row.append(str(history.history[metric][-1]))
    df.loc[len(df)] = lst_row
    
    df.to_excel(all_results_path, index=False)
    
    print('Writing train predictions...')
    write_predictions(model, x_train, y_train, original_labels, dir_name, 'train')
    print('Writing val predictions...')
    write_predictions(model, x_val, y_val, original_labels, dir_name, 'val')
    print('Writing test predictions...')
    write_predictions(model, x_test, y_test, original_labels, dir_name, 'test')
    
    return history, x_test, y_test, y_pred_test, y_pred_classes_test, model
    

In [16]:
history, x_test, y_test, y_pred_test, y_pred_classes_test, model = play(df, **dict_hyperparams)

Play function called with hyperparameters: {'lookback_window': 25, 'batch_size': 32, 'epochs': 1, 'nb_filters': 64, 'nb_stacks': 10, 'kernel_size': 3, 'use_batch_norm': True, 'activation': 'relu', 'optimizer': 'adamw'}


I0000 00:00:1779380575.950500   18509 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_209947__.633
W0000 00:00:1779380589.216678   19785 nvptx_backend.cc:112] libdevice is required by this HLO module but was not found at ./libdevice.10.bc
W0000 00:00:1779380589.417640   18509 op_kernel.cc:1858] OP_REQUIRES failed at xla_ops.cc:602 : INTERNAL: Autotuner could not compile any configs for HLO: %loop_add_subtract_fusion.140 = (f32[64]{0}, f32[64]{0}, f32[64]{0}) fusion(%get-tuple-element.5682, %arg3.1, %loop_multiply_fusion.1, %arg732.1, %get-tuple-element.4434, /*index=5*/%arg733.1), kind=kLoop, calls=%fused_add_subtract.140, metadata={op_type="AssignSubVariableOp" op_name="adamw/AssignSubVariableOp_485" source_file="/mnt/d/work/cercetare/2026/paper_Stelian/code/source/.venv/lib/python3.13/site-packages/tensorflow/python/framework/ops.py" source_line=1221 deduplicated_name="loop_add_subtract_fusion.20"}
W0000 00:00:1779380589.419739   18509 local_rendezvous.cc

InternalError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/mnt/d/work/cercetare/2026/paper_Stelian/code/source/.venv/lib/python3.13/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/mnt/d/work/cercetare/2026/paper_Stelian/code/source/.venv/lib/python3.13/site-packages/traitlets/config/application.py", line 1082, in launch_instance

  File "/mnt/d/work/cercetare/2026/paper_Stelian/code/source/.venv/lib/python3.13/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/mnt/d/work/cercetare/2026/paper_Stelian/code/source/.venv/lib/python3.13/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/home/lucian/.local/share/uv/python/cpython-3.13.11-linux-x86_64-gnu/lib/python3.13/asyncio/base_events.py", line 683, in run_forever

  File "/home/lucian/.local/share/uv/python/